# Chapter 1 — Reading data · Solution

*Maths for AI, Step by Step*, Belk Siri — companion notebook.

[Open in Google Colab](https://colab.research.google.com/github/belksiri/books/blob/main/maths-pour-l-ia-pas-a-pas/notebooks/en/ch01_corrige.ipynb) · Companion page: https://belksiri.github.io/books/maths-pour-l-ia-pas-a-pas/en.html

Each exercise restates the book's problem. The answer starts with the reasoning on paper, then shows its equivalent in Python.

**Convention.** As in the book, the variance, the standard deviation, the z-score and the covariance divide by $n$ (`ddof=0`), unless stated otherwise.

**Contents.** Data · The lesson calculations · Exercises 1.1 to 1.6

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (7, 2.6)
# The book's colors: one color = one role (observed, predicted, deviation, colleague's line)
BLUE, ORANGE, RESIDUAL, GREEN = '#2a78d6', '#eb6834', '#c2185b', '#2e8b57'

## Data

The identifiers (D0.1, D1.1…) are those of the book and of the CSV files.

In [ ]:
# Chapter 1 data, copied from the book's CSV files (do not modify)

# D0.1 — the six homes of the opening
homes = pd.DataFrame({
    'home':   ['A', 'B', 'C', 'D', 'E', 'F'],
    'floor_area': [32, 45, 54, 70, 85, 98],
    'price':  [118, 172, 185, 245, 279, 335],
})

# D1.1 — sale prices of the T2 apartments (two rooms, k€), lesson 1.1
north = [174, 177, 181, 183, 185]
south  = [150, 165, 185, 195, 205]

# D1.2 — register of sales in the South district, lessons 1.2 to 1.5
sales_log = pd.DataFrame({
    'sale':       ['S01', 'S02', 'S03', 'S04', 'S05', 'S06', 'S07', 'S08', 'S09', 'S10', 'S11', 'S12', 'S13'],
    'rooms':      [1, 1, 2, 2, 2, 2, 2, 3, 3, 3, 4, 4, 4],
    'floor_area':  [24, 27, 43, 40, 44, 46, 52, 61, 64, 70, 82, 88, 95],
    'energy_class':         ['D', 'C', 'F', 'D', 'C', 'D', 'C', 'E', 'D', 'B', 'E', 'C', 'G'],
    'price':   [96, 108, 150, 165, 185, 195, 205, 228, 241, 276, 289, 318, 260],
    'days_to_sell': [9, 18, 97, 21, 26, 23, 31, 14, 35, 28, 42, 58, 183],
})

# D1.5 — price per m² by floor (k€/m²), lesson 1.5
floors = pd.DataFrame({
    'floor':        [0, 1, 2, 3, 4, 5, 6],
    'price_per_m2': [3.0, 3.3, 3.5, 3.6, 3.5, 3.3, 3.0],
})

# D1.E1 — selling times of three agencies (days), exercise 1.1
agency_delays = {'P': [28, 29, 30, 31, 32], 'Q': [20, 25, 30, 35, 40], 'R': [10, 28, 30, 32, 50]}

# D1.E2 — monthly rents of the studios (€), exercise 1.2
tilleuls = [585, 590, 600, 605, 605, 615]
port     = [520, 560, 590, 620, 640, 670]

### Data variables

Code names are in English. For the columns, the last column gives the name used in the book's CSV files.

| Name | Meaning | Unit | CSV column |
|:--|:--|:--|:--|
| `home` | home identifier (A, B…) | — | `logement` |
| `floor_area` | living area of the home | m² | `surface_m2` |
| `price` | sale price | k€ | `prix_kEUR` |
| `sale` | sale identifier (S01…) | — | `vente` |
| `rooms` | number of rooms | rooms | `pieces` |
| `energy_class` | energy performance rating class (A to G) | — | `dpe` |
| `days_to_sell` | time to sell | days | `delai_jours` |
| `floor` | floor | — | `etage` |
| `price_per_m2` | price per square meter | k€/m² | `prix_m2_kEUR` |

In [ ]:
# Tools provided: you do not need to modify them

def book_quartiles(values):
    """Q1 and Q3 by the book's rule (equation 1.5): ranks ⌈n/4⌉ and ⌈3n/4⌉."""
    x = sorted(values)
    n = len(x)
    rank_q1 = -(-n // 4)          # n/4 rounded up to the next integer
    rank_q3 = -(-3 * n // 4)      # 3n/4 rounded up to the next integer
    return x[rank_q1 - 1], x[rank_q3 - 1]   # rank 1 = position 0 in Python


def dot_rows(series, xlabel, bands=None, ax=None):
    """One row of dots per series, as in the figures of lesson 1.1.
    bands: dictionary {name: (mean, standard deviation)} to draw mean ± standard deviation."""
    if ax is None:
        fig, ax = plt.subplots()
    names = list(series)
    colors = [BLUE, ORANGE, '#52514e']
    for i, name in enumerate(names):
        y = len(names) - 1 - i
        if bands and name in bands:
            m, s = bands[name]
            ax.add_patch(plt.Rectangle((m - s, y - 0.25), 2 * s, 0.5, color='#e9e8e4'))
            ax.plot([m, m], [y - 0.3, y + 0.3], color='black')
        ax.scatter(series[name], [y] * len(series[name]), s=50,
                   color=colors[i % 3], zorder=3)
    ax.set_yticks(range(len(names)), list(reversed(names)))
    ax.set_ylim(-0.6, len(names) - 0.4)
    ax.set_xlabel(xlabel)
    ax.grid(axis='x', alpha=0.4)
    return ax

## The lesson calculations, in Python

These cells redo the calculations of lessons 1.1 to 1.5. Run them and compare with the book. Here, `np.std` and `np.var` are called with `ddof=0`, that is, with a division by $n$, as in the chapter.

In [ ]:
# Lesson 1.1 — mean, variance and standard deviation (division by n)
for name, one_series in [('North', north), ('South', south)]:
    print(f'{name}: mean = {np.mean(one_series):g} k€, variance = {np.var(one_series, ddof=0):g} k€², '
          f'standard deviation = {np.std(one_series, ddof=0):g} k€')

In [ ]:
# Lesson 1.2 — counts per energy rating class (ordinal qualitative variable)
sales_log['energy_class'].value_counts().reindex(list('ABCDEFG'), fill_value=0)

In [ ]:
# Lesson 1.3 — mean, median and quartiles of the selling times
d = sales_log['days_to_sell']
print('mean:', d.mean(), ' median:', d.median())
print('without S13: mean', d[sales_log['sale'] != 'S13'].mean(),
      ' median', d[sales_log['sale'] != 'S13'].median())
print("quartiles, the book's rule (equation 1.5):", book_quartiles(d))
print('quartiles, NumPy default (interpolation):', np.quantile(d, [0.25, 0.75]))

In [ ]:
# Lesson 1.4 — standardized scores (equation 1.6, standard deviation with division by n)
for name, one_series, sale in [('North', north, 188), ('South', south, 210)]:
    m, s = np.mean(one_series), np.std(one_series, ddof=0)
    print(f'{name}: z of the series = {np.round((np.array(one_series) - m) / s, 4)}, '
          f'z of the sale at {sale} k€ = {(sale - m) / s:g}')

In [ ]:
# Lesson 1.5 — covariance and correlation of the five T2 apartments in the South (division by n)
t2 = sales_log[sales_log['rooms'] == 2]
x, y = t2['floor_area'].to_numpy(float), t2['price'].to_numpy(float)
cov = np.mean((x - x.mean()) * (y - y.mean()))
print('covariance:', cov, 'm²·k€   r:', cov / (x.std(ddof=0) * y.std(ddof=0)))
print('r from np.corrcoef:', np.corrcoef(x, y)[0, 1].round(4))
print('r floor / price per m²:', np.corrcoef(floors['floor'], floors['price_per_m2'])[0, 1].round(4))

## Exercise 1.1 — Reading a chart

*Data D1.E1. Concepts: mean and standard deviation (lesson 1.1).* The chart below reproduces the one in exercise 1.1 of the book.

In [ ]:
dot_rows({f'Agency {a}': v for a, v in agency_delays.items()},
                  'Selling time (days)')
plt.xlim(0, 60)
plt.show()

**a) and b)** Estimate by eye the mean selling time of each agency, then rank the agencies from the least spread out to the most spread out.

In [ ]:
estimated_mean = {'P': 30, 'Q': 30, 'R': 30}   # in days
spread_order = ['P', 'Q', 'R']

In [ ]:
# Check (division by n)
for a, v in agency_delays.items():
    print(f'Agency {a}: mean = {np.mean(v):g} d, standard deviation = {np.std(v, ddof=0):.2f} d')
true_order = sorted(agency_delays, key=lambda a: np.std(agency_delays[a]))
if spread_order is None or None in estimated_mean.values():
    print('Complete the previous cell.')
else:
    print('Spread order:', 'correct' if spread_order == true_order else 'check again')

**c)** For Q and for R, count the sales closed within 2 days or less of the mean.

In [ ]:
close_to_Q = 1
close_to_R = 3

In [ ]:
# Check
for a, yours in [('Q', close_to_Q), ('R', close_to_R)]:
    v = np.array(agency_delays[a])
    truth = int(np.sum(np.abs(v - v.mean()) <= 2))
    if yours is None:
        print(f'Agency {a}: not yet completed.')
    else:
        print(f"Agency {a}: {'correct' if yours == truth else 'check again'}")

### Solution to exercise 1.1

**a)** The three series are symmetric around 30 days. The mean selling time is **30 days** for each agency, as the calculation confirms: (28 + 29 + 30 + 31 + 32) / 5 = 150 / 5 = 30, and likewise for Q and R.

**b)** From least to most spread out: **P, Q, R**. The dots of P fit between 28 and 32 days. Those of Q spread evenly from 20 to 40 days. Those of R range from 10 to 50 days. Standard deviations (division by $n$): P ≈ 1.41 d; Q ≈ 7.07 d; R ≈ 12.71 d.

**c)** Q: **1** sale (30 days). R: **3** sales (28, 30 and 32 days). Yet R has the largest standard deviation. Its two extreme sales, at 10 and 50 days, are 20 days from the mean. Their squares are 400 each: the sum of squares of R is 400 + 4 + 0 + 4 + 400 = 808, against 100 + 25 + 0 + 25 + 100 = 250 for Q. The standard deviation summarizes all the deviations, and squaring gives a large weight to the biggest ones. Three sales close to the center do not make up for two sales very far away.

*Expected elements:* the ranking P, Q, R; the two counts; the role of the two extreme values and of the squares in the calculation.

## Exercise 1.2 — Two residences, the same mean rent

*Data D1.E2. Concepts: mean, variance and standard deviation (lesson 1.1).* Camille cannot spend more than 570 € a month. Questions a) to d) are first done on paper, with the table of exercise 1.2 in the book. This notebook lets you check your answers, then program the standard deviation (question e) and compare two conventions (question f).

In [ ]:
dot_rows({'Les Tilleuls': tilleuls, 'Le Port': port}, "Monthly rent of a studio (€)")
plt.axvline(570, color='black', linestyle=':', label="Camille's budget: 570 €")
plt.xlim(500, 700)
plt.legend(loc='upper right', fontsize=8)
plt.show()

### a) Reading the rents

How many studios can Camille rent in each residence with 570 €, then with 620 €?

In [ ]:
affordable = {
    570: {'Les Tilleuls': 0, 'Le Port': 2},
    620: {'Les Tilleuls': 6, 'Le Port': 4},
}

In [ ]:
# Check: `<=` produces true/false values, np.sum counts the "true" ones
for budget, answers in affordable.items():
    for name, one_series in [('Les Tilleuls', tilleuls), ('Le Port', port)]:
        truth = int(np.sum(np.array(one_series) <= budget))
        yours = answers[name]
        status = 'not yet completed' if yours is None else ('correct' if yours == truth else 'check again')
        print(f'Budget {budget} €, {name}: {status}')

### Solution to 1.2 a)

**Reading the list.** With 570 €, Camille finds **no** studio at Les Tilleuls: the cheapest one costs 585 €. At Le Port, **two** studios out of six are within her reach, at 520 and 560 €. So the residence where she has a chance is the one whose rents are the most spread out. With 620 €, the situation is reversed: all **six** studios at Les Tilleuls are within her reach, against **four** out of six at Le Port (520, 560, 590 and 620 €). The two equal means gave no hint of these differences.

### b) The two means

**On paper.** Les Tilleuls: 585 + 590 + 600 + 605 + 605 + 615 = 3,600, and 3,600 / 6 = **600 €**. Le Port: 520 + 560 + 590 + 620 + 640 + 670 = 3,600, and 3,600 / 6 = **600 €**.

**In Python.** `np.mean` does the same sum, then the same division.

In [ ]:
mean_tilleuls_by_hand = 600
mean_port_by_hand = 600

In [ ]:
# Check
for name, one_series, yours in [('Les Tilleuls', tilleuls, mean_tilleuls_by_hand),
                              ('Le Port', port, mean_port_by_hand)]:
    if yours is None:
        print(f'{name}: complete the previous cell.')
    else:
        ok = np.isclose(yours, np.mean(one_series))
        print(f"{name}: {'correct' if ok else 'check again'} (your value: {yours} €)")

### c) Variance and standard deviation by hand (division by $n = 6$)

**Les Tilleuls** ($\bar{x} = 600$ €)

| $x_i$ (€) | 585 | 590 | 600 | 605 | 605 | 615 | Sum |
|:--|--:|--:|--:|--:|--:|--:|--:|
| $x_i - \bar{x}$ (€) | −15 | −10 | 0 | +5 | +5 | +15 | 0 |
| $(x_i - \bar{x})^2$ (€²) | 225 | 100 | 0 | 25 | 25 | 225 | 600 |

Variance: 600 / 6 = **100 €²**. Standard deviation: √100 = **10 €**.

**Le Port** ($\bar{x} = 600$ €)

| $x_i$ (€) | 520 | 560 | 590 | 620 | 640 | 670 | Sum |
|:--|--:|--:|--:|--:|--:|--:|--:|
| $x_i - \bar{x}$ (€) | −80 | −40 | −10 | +20 | +40 | +70 | 0 |
| $(x_i - \bar{x})^2$ (€²) | 6,400 | 1,600 | 100 | 400 | 1,600 | 4,900 | 15,000 |

Variance: 15,000 / 6 = **2,500 €²**. Standard deviation: √2,500 = **50 €**.

*Two checks along the way.* In each table, the row of deviations must add up to 0 (lesson 1.1). Next, the standard deviation must stay of the same order as the deviations in the table: 10 € for deviations of 0 to 15 €, 50 € for deviations of 10 to 80 €.

**In Python.** `np.var(..., ddof=0)` and `np.std(..., ddof=0)` divide by $n$.

In [ ]:
variance_tilleuls_by_hand = 100   # in €²
variance_port_by_hand = 2500      # in €²
std_tilleuls_by_hand = 10  # in €
std_port_by_hand = 50      # in €

In [ ]:
# Check (division by n: ddof=0)
your_values = {
    'variance Tilleuls': (variance_tilleuls_by_hand, np.var(tilleuls, ddof=0)),
    'variance Port': (variance_port_by_hand, np.var(port, ddof=0)),
    'standard deviation Tilleuls': (std_tilleuls_by_hand, np.std(tilleuls, ddof=0)),
    'standard deviation Port': (std_port_by_hand, np.std(port, ddof=0)),
}
for name, (yours, expected) in your_values.items():
    if yours is None:
        print(f'{name}: not yet completed.')
    else:
        print(f"{name}: {'correct' if np.isclose(yours, expected) else 'check again'}")

### d) Interpreting

*Sample answer.* At Les Tilleuls, rents usually differ by about 10 € from 600 €: they are almost all the same. At Le Port, the usual deviation is about 50 €: there are studios clearly cheaper and clearly more expensive than the mean. The standard deviation is not enough to answer question a). It describes the spread in general, but does not say whether a particular studio costs 570 € or less. A standard deviation of 50 € around 600 € makes a rent below 570 € plausible. Only the list lets you count that there are exactly two, at 520 and 560 €.

*Expected elements:* one sentence per residence, with the unit; the distinction between a measure of spread and the question "does a studio fit the budget?"; the use of the list of rents.

### e) Programming the standard deviation

Complete the function by following equation 1.3 of the book, with a division by $n$:

$$\sigma = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (x_i - \bar{x})^2}$$

Use neither `np.std` nor `np.var`: the goal is to write each step yourself.

In [ ]:
# Exercise 1.2 e)
def std_dev(values):
    """Standard deviation of a series, with division by n (equation 1.3)."""
    x = np.array(values, dtype=float)
    n = len(x)
    mean = np.sum(x) / n               # equation 1.1
    deviations = x - mean                  # one subtraction per value
    variance = np.sum(deviations ** 2) / n    # equation 1.2
    return np.sqrt(variance)              # back to euros

Each line of the function matches a column of the table in question c): the deviations, their squares, the mean of these squares, then the square root to get back to euros. Common mistakes: taking the root before the mean; computing `np.sum(deviations) / n`, which always gives 0 (lesson 1.1).

In [ ]:
# Check: run this cell without modifying it
if std_dev(tilleuls) is None:
    print('The function still returns None: complete it.')
else:
    for name, one_series, expected in [('Les Tilleuls', tilleuls, 10.0), ('Le Port', port, 50.0)]:
        obtained = std_dev(one_series)
        status = 'correct' if np.isclose(obtained, expected) else f'check again (expected: {expected})'
        print(f'{name}: {obtained:.2f} € → {status}')

### f) Going further: dividing by $n$ or by $n - 1$

By default, the pandas `.std()` method divides the sum of squares by $n - 1$ (`ddof=1`). NumPy divides by $n$ by default (`ddof=0`), the chapter's convention.

**On paper, with division by $n - 1 = 5$.** Tilleuls: 600 / 5 = 120, then √120 ≈ **10.95 €**. Port: 15,000 / 5 = 3,000, then √3,000 ≈ **54.77 €**.

Both conventions are correct, but they do not answer the same question. Dividing by $n$ describes the six studios themselves. Dividing by $n - 1$ is used when these six studios represent a larger set. Chapter 6 will explain this choice. With `.std(ddof=0)`, pandas finds 10 and 50 € again. In a calculation, always state the convention you use.

In [ ]:
rents = pd.DataFrame({'Les Tilleuls': tilleuls, 'Le Port': port})
print('pandas, default (division by n − 1):', rents.std().round(2).to_dict())
print('pandas, ddof=0 (division by n)         :', rents.std(ddof=0).round(2).to_dict())

## Exercise 1.3 — A surprising sale?

*Data D1.1. Concepts: standardized score (lesson 1.4, equation 1.6).* A T2 (two-room apartment) has just sold for 196 k€. Means and standard deviations: those of lesson 1.1, with division by $n$.

In [ ]:
z_north = (196 - 180) / 4     # = 4
z_south = (196 - 180) / 20     # = 0.8
south_price_as_unusual = 180 + 4 * 20   # = 260 k€

In [ ]:
# Check (standard deviation with division by n)
expected_values = {
    'z North': (z_north, (196 - np.mean(north)) / np.std(north, ddof=0)),
    'z South': (z_south, (196 - np.mean(south)) / np.std(south, ddof=0)),
    'equivalent South price': (south_price_as_unusual,
                            np.mean(south) + (196 - np.mean(north)) / np.std(north, ddof=0) * np.std(south, ddof=0)),
}
for name, (yours, truth) in expected_values.items():
    status = 'not yet completed' if yours is None else ('correct' if np.isclose(yours, truth) else 'check again')
    print(f'{name}: {status}')

### Solution to exercise 1.3

**a)** In the North: $z = (196 - 180)/4 = 16/4 =$ **4**. In the South: $z = (196 - 180)/20 = 16/20 =$ **0.8**. Both scores have no unit: they count standard deviations.

**b)** It would be surprising in the **North**. On the figure of standardized scores in lesson 1.4, a score of 4 falls well beyond all the sales in the North, none of which exceeds 1.25. In the South, 0.8 lies between the sales at 0.75 and 1.25: an ordinary sale.

**c)** We look for the South price whose score is 4: $180 + 4 \times 20 =$ **260 k€**. Go back to equation 1.6 and isolate $x$: $x = \bar{x} + z\,\sigma$.

*Expected elements:* the two scores; the comparison in standard deviations rather than in k€; the equivalent price of 260 k€.

## Exercise 1.4 — Do floor area and price go together?

*Data D0.1 (the six homes of the opening). Concepts: covariance and correlation (lesson 1.5, equations 1.7 and 1.8).*

In [ ]:
plt.figure(figsize=(5, 3.5))
plt.scatter(homes['floor_area'], homes['price'], color=BLUE)
for _, l in homes.iterrows():
    plt.annotate(l['home'], (l['floor_area'], l['price']), xytext=(-10, 4), textcoords='offset points')
plt.xlabel('Floor area (m²)')
plt.ylabel('Sale price (k€)')
plt.grid(alpha=0.4)
plt.show()

**a)** The dots rise from left to right and almost follow a line: $r$ will be **positive and very close to 1**.

**b)** Complete the lines below. Use a division by $n$ everywhere.

In [ ]:
x = homes['floor_area'].to_numpy(dtype=float)
y = homes['price'].to_numpy(dtype=float)
n = len(x)

products = (x - x.mean()) * (y - y.mean())          # one product per home
covariance = np.sum(products) / n                   # equation 1.7
r = covariance / (x.std(ddof=0) * y.std(ddof=0))    # equation 1.8

**On paper**, the mean floor area is 384 / 6 = 64 m² and the mean price 1,334 / 6 ≈ 222.33 k€. The sum of the six products is 9,825, hence a covariance of 9,825 / 6 = **1,637.5 m²·k€**. The standard deviations are about 22.80 m² and 72.15 k€, so $r ≈ 1637.5 / (22.80 × 72.15) ≈$ **0.996**.

In [ ]:
# Check
if r is None or covariance is None:
    print('Complete the previous cell.')
else:
    print('covariance:', round(covariance, 2), 'm²·k€ →',
          'correct' if np.isclose(covariance, 1637.5) else 'check again (expected: 1637.5)')
    print('r:', round(r, 4), '→', 'correct' if np.isclose(r, np.corrcoef(x, y)[0, 1]) else 'check again')

**A trap to know about.** `np.cov(x, y)` divides by $n - 1$ by default and returns 1,965, not 1,637.5. For the chapter's convention, you must write `np.cov(x, y, bias=True)`. The coefficient $r$, on the other hand, does not depend on the convention, because the same factor appears at the top and at the bottom (lesson 1.5).

**c)** There is no contradiction. First, $r$ measures how tightly the dots cluster around *the* line that fits them best (the one chapter 3 will teach you to compute), not around our line drawn by eye. Next, closeness is relative: prices spread with a standard deviation of about 72 k€, and deviations of 5 to 11 k€ in absolute value are small next to this spread. So an $r$ close to 1 does not guarantee small errors in k€. To know the size of the errors, you have to compute them, as in exercise 1.6.

*Expected elements:* "$r$ is about the best line"; "the deviations are small relative to the spread of prices"; the distinction between alignment and size of the errors in k€.

In [ ]:
print('np.cov by default (division by n − 1):', np.cov(x, y)[0, 1])
print('np.cov with bias=True (division by n) :', np.cov(x, y, bias=True)[0, 1])

## Exercise 1.5 — Predict, then check

*Data D1.1. Concepts: mean and standard deviation (lesson 1.1), median and interquartile range (lesson 1.3).* A sixth T2 sells in the North for 240 k€.

**a)** Expected prediction: the mean rises clearly, because it spreads the 240 k€ over all the sales. The median barely moves, because the new sale simply takes its place "after the middle". The standard deviation rises a lot, because a deviation of about 50 k€, once squared, dominates the sum. The interquartile range moves little, because it depends only on the central values.

In [ ]:
north_plus = north + [240]
print('North:', sorted(north))
print('North with the sale at 240 k€:', sorted(north_plus))

**b)** Complete the function `summary`. It must return the four indicators, with the standard deviation divided by $n$ and the quartiles computed by the book's rule (`book_quartiles`, provided above). The dictionary keys are `mean`, `median`, `std_dev` (standard deviation) and `iqr` (interquartile range).

In [ ]:
def summary(one_series):
    q1, q3 = book_quartiles(one_series)
    return {
        'mean': np.mean(one_series),
        'median': np.median(one_series),
        'std_dev': np.std(one_series, ddof=0),   # division by n
        'iqr': q3 - q1,
    }

In [ ]:
table = pd.DataFrame({'North (5 sales)': summary(north), 'North+240 k€ (6 sales)': summary(north_plus),
                        'South (5 sales)': summary(south)})
if table.isna().any().any():
    print('Complete the function summary.')
else:
    expected = {'mean': (180, 190), 'median': (181, 182), 'std_dev': (4, 22.66),
               'iqr': (6, 8)}
    for k, (before, after) in expected.items():
        ok = np.isclose(table.loc[k].iloc[0], before) and np.isclose(table.loc[k].iloc[1], after, atol=0.005)
        print(f"{k}: {'correct' if ok else 'check again'}")
table.round(2)

In [ ]:
# The two series, modeled on the "mean ± standard deviation" figure of lesson 1.1
if table.isna().any().any():
    print('Complete the function summary to display the chart.')
else:
    dot_rows({'North': north, 'North + 240': north_plus},
                      'Sale price of a T2 (k€)',
                      bands={'North': (np.mean(north), np.std(north, ddof=0)),
                              'North + 240': (np.mean(north_plus), np.std(north_plus, ddof=0))})
    plt.xlim(140, 250)
    plt.show()

### Solution to exercise 1.5

**b) On paper.** With the sale at 240 k€, the sum goes from 900 to 1,140 k€ for 6 sales: mean **190 k€** (instead of 180). Sorted, the six values are 174, 177, 181, 183, 185, 240. $n$ is even, so the median is (181 + 183)/2 = **182 k€** (instead of 181). The deviations from 190 are −16, −13, −9, −7, −5, +50. Their squares are 256, 169, 81, 49, 25 and 2,500, for a sum of 3,080. The variance is 3,080/6 ≈ 513.3 k€², and the standard deviation **≈ 22.66 k€** (instead of 4). Quartiles (equation 1.5): before, ranks ⌈5/4⌉ = 2 and ⌈15/4⌉ = 4, i.e. Q1 = 177 and Q3 = 183, hence an interquartile range of **6 k€**. After, ranks ⌈6/4⌉ = 2 and ⌈18/4⌉ = 5, i.e. Q1 = 177 and Q3 = 185, hence **8 k€**.

**c)** No. Five sales out of six remain within 6 k€ or less of 180 k€. A single exceptional sale is enough to push the standard deviation from 4 to 22.66 k€, above the 20 k€ of the South, because its deviation of 50 k€ counts for 2,500 in a sum of 3,080. The robust measures tell another story: the interquartile range of the North goes from 6 to 8 k€ and stays far below that of the South (30 k€). For an ordinary T2 in the North, prices remain very predictable. The standard deviation mainly signals the presence of an unusual sale, which must be looked at separately.

*Expected elements:* the four values before/after; the contrast between sensitive indicators (mean, standard deviation) and robust ones (median, interquartile range); a nuanced conclusion.

## Exercise 1.6 — Challenge: two lines, the same mean error

*Data D0.1. Concepts: signed deviation and absolute value (opening, lesson 1.1), standard deviation (lesson 1.1).* Line drawn by eye: predicted price = 3 × floor area + 30. Lazy line: predicted price = 222 k€ for every home.

In [ ]:
price = homes['price'].to_numpy(dtype=float)
floor_area = homes['floor_area'].to_numpy(dtype=float)

eye_line_errors = price - (3 * floor_area + 30)
lazy_line_errors = price - 222

In [ ]:
# The three indicators of question b) (standard deviation: division by n)
if eye_line_errors is None or lazy_line_errors is None:
    print('Complete the previous cell.')
else:
    indicators = pd.DataFrame({
        name: {'mean of signed deviations': e.mean(),
              'mean of absolute deviations': np.abs(e).mean(),
              'standard deviation of signed deviations': e.std(ddof=0)}
        for name, e in [('line by eye', eye_line_errors), ('lazy line', lazy_line_errors)]
    })
    print('deviations, line by eye:', eye_line_errors)
    print('deviations, lazy line :', lazy_line_errors)
    print('check:', 'correct' if np.allclose(indicators.iloc[:, 0], [1/3, 44/6, 7.5645], atol=1e-3)
          and np.allclose(indicators.iloc[:, 1], [1/3, 64, 72.1472], atol=1e-3) else 'check again')
    display(indicators.round(2))

In [ ]:
if eye_line_errors is not None and lazy_line_errors is not None:
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.axhline(0, color='black', linewidth=1)
    ax.scatter(floor_area - 0.8, eye_line_errors, color=ORANGE, label="line by eye")
    ax.scatter(floor_area + 0.8, lazy_line_errors, color=BLUE, marker='s', label='lazy line')
    ax.set_xlabel('Floor area (m²)')
    ax.set_ylabel('Deviation observed − predicted (k€)')
    ax.legend()
    ax.grid(alpha=0.4)
    plt.show()

### Solution to exercise 1.6

**a)** Deviations of the lazy line (observed − 222): −104, −50, −37, +23, +57, +113 k€. Their sum is 2 and their mean ≈ **0.33 k€**. This is exactly the mean of the signed deviations of the line drawn by eye (−8, +7, −7, +5, −6, +11: sum 2, mean ≈ 0.33 k€).

**b)**

| | Line by eye | Lazy line |
|:--|--:|--:|
| Mean of signed deviations | 0.33 k€ | 0.33 k€ |
| Mean of absolute deviations | 44/6 ≈ 7.33 k€ | 384/6 = 64 k€ |
| Standard deviation of signed deviations (division by $n$) | ≈ 7.56 k€ | ≈ 72.15 k€ |

Detail for the line by eye: the mean is 1/3. The deviations from this mean have squares whose sum is 344 − 6 × (1/3)² ≈ 343.33. The variance is 343.33/6 ≈ 57.22 and the standard deviation ≈ 7.56 k€.

**c)** The line drawn by eye is much better: its errors are about 7 k€ in absolute value, against 64 k€ for the lazy line. The mean of absolute deviations and the standard deviation of signed deviations tell them apart. The mean of signed deviations does not: neither line leans to one side, and the signs cancel out in both cases.

**d)** The deviations of the lazy line are $y_i - 222$. Subtracting the same constant shifts the whole series without changing the distance of each value from the mean. The mean also decreases by 222, so each deviation from the mean $(y_i - 222) - (\bar{y} - 222) = y_i - \bar{y}$ stays the same. The standard deviation of the deviations is therefore that of the prices, ≈ 72.15 k€. A line that ignores floor area is wrong, on average, by the spread of the prices itself. Chapter 3 will make use of this comparison.

*Expected elements:* the two equal means; two clearly different indicators of error size; the explanation by the canceling of signs; the shift argument for d).

In [ ]:
# Check of d): the standard deviation of the lazy line's deviations equals that of the prices
if lazy_line_errors is not None:
    print(round(lazy_line_errors.std(ddof=0), 4), round(price.std(ddof=0), 4))